# flatten_param_tuples
考虑有 $K$ 个类型的参数，每个类型的参数给出了 $N$ 个取法：
$$parm_1^1, \cdots ,parm_1^N, \cdots ,parm_K^1, \cdots parm_K^N$$
函数的作用就是转换为
$$\left[ {\left[ {parm_1^1 \cdots ,parm_K^1} \right] \cdots \left[ {parm_1^N \cdots ,parm_K^N} \right]} \right]$$


```python
def flatten_param_tuples(param_tuples: tp.Sequence) -> tp.List[tp.List]:
    param_list = []
    unzipped_tuples = zip(*param_tuples)
    for i, unzipped in enumerate(unzipped_tuples):
        unzipped = list(unzipped)
        if isinstance(unzipped[0], tuple):
            param_list.extend(flatten_param_tuples(unzipped))
        else:
            param_list.append(unzipped)
    return param_list
```

## 例子

In [2]:
from vectorbt.utils.params import flatten_param_tuples

weight_params = [
     ((0.3, 0.7), (0.4, 0.6)),  # 股票权重
     ((0.2, 0.8), (0.3, 0.7))   # 债券权重  
]
weights = flatten_param_tuples(weight_params)
print(weights)

[[0.3, 0.2], [0.7, 0.8], [0.4, 0.3], [0.6, 0.7]]


# create_param_combs
基于操作树 `op_tree` 进行参数组合
- `op_tree[0]` 是可调用对象，`op_tree[1:]` 是该函数的参数
- 如果 `op_tree[1:]` 的某元素也是递归树，会递归处理
- 如果 `depth == 0`，最后会调用 `flatten_param_tuples`

```python
def create_param_combs(op_tree: tp.Tuple, depth: int = 0) -> tp.List[tp.List]:
    checks.assert_instance_of(op_tree, tuple)
    checks.assert_instance_of(op_tree[0], Callable)
    new_op_tree: tp.Tuple = (op_tree[0],)
    for elem in op_tree[1:]:
        if isinstance(elem, tuple) and isinstance(elem[0], Callable):
            new_op_tree += (create_param_combs(elem, depth=depth + 1),)
        else:
            new_op_tree += (elem,)
    out = list(new_op_tree[0](*new_op_tree[1:]))
    if depth == 0:
        return flatten_param_tuples(out)
    return out
```

## 例子

In [10]:
from vectorbt.utils.params import create_param_combs
from itertools import product

# 自定义参数生成函数
def custom_ranges(start, end, step):
    return list(range(start, end, step))

custom_params = create_param_combs(
     (product, (custom_ranges, 1, 4, 2), [0.01, 0.02, 0.03])
)

print(len(custom_params))
print(len(custom_params[0]))
print(custom_params)

2
6
[[1, 1, 1, 3, 3, 3], [0.01, 0.02, 0.03, 0.01, 0.02, 0.03]]


# broadcast_params
将参数列表序列 `param_list` 广播到统一长度 `to_n`，并组合为一个新的 `List`。

```python
def broadcast_params(param_list: tp.Sequence[tp.Sequence], to_n: tp.Optional[int] = None) -> tp.List[tp.List]:
    if to_n is None:
        to_n = max(list(map(len, param_list)))
    new_param_list = []
    for i in range(len(param_list)):
        params = param_list[i]
        if len(params) in [1, to_n]:
            if len(params) < to_n:
                new_param_list.append([p for _ in range(to_n) for p in params])
            else:
                new_param_list.append(list(params))
        else:
            raise ValueError(f"Parameters at index {i} have length {len(params)} that cannot be broadcast to {to_n}")
    return new_param_list
```

## 例子

In [12]:
from vectorbt.utils.params import broadcast_params

fees = [0.001]  # 单一手续费率
sizes = [100, 200, 300]  # 不同资产的交易数量
symbols = ['AAPL', 'GOOGL', 'MSFT']  # 股票代码
 
broadcasted = broadcast_params([fees, sizes, symbols])
print(broadcasted)

[[0.001, 0.001, 0.001], [100, 200, 300], ['AAPL', 'GOOGL', 'MSFT']]


# create_param_product
返回参数列表序列 `param_list` 的笛卡尔积。


```python
def create_param_product(param_list: tp.Sequence[tp.Sequence]) -> tp.List[tp.List]:
    return list(map(list, zip(*list(itertools.product(*param_list)))))
```

## 例子

In [16]:
from vectorbt.utils.params import create_param_product

fast_periods = [5, 10, 15]    # 快速均线周期
slow_periods = [20, 30, 40]   # 慢速均线周期
thresholds = [0.01, 0.02]     # 交易阈值
 
combinations = create_param_product([fast_periods, slow_periods, thresholds])
print(f"结果的行数为：{len(combinations)}，列数为：{len(combinations[0])}")
print(combinations[0])
print(combinations[1])
print(combinations[2])

结果的行数为：3，列数为：18
[5, 5, 5, 5, 5, 5, 10, 10, 10, 10, 10, 10, 15, 15, 15, 15, 15, 15]
[20, 20, 30, 30, 40, 40, 20, 20, 30, 30, 40, 40, 20, 20, 30, 30, 40, 40]
[0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02, 0.01, 0.02]
